In [ ]:
import time
import warnings
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

X = pd.read_csv("../data/secom.data", sep=" ", header=None)
y = pd.read_csv("../data/secom_labels.data", sep=" ", header=None,
                names=["label", "timestamp"])
y["timestamp"] = pd.to_datetime(y["timestamp"], format="%d/%m/%Y %H:%M:%S")

df = X.copy()
df["label"] = (y["label"] == 1).astype(int)
df["timestamp"] = y["timestamp"]
df = df.sort_values("timestamp").reset_index(drop=True)

split = int(len(df) * 0.8)
train, test = df.iloc[:split], df.iloc[split:]

feature_cols = [c for c in df.columns if c not in ("label", "timestamp")]
X_train, y_train = train[feature_cols], train["label"]

keep = [c for c in feature_cols
        if X_train[c].isna().mean() <= 0.5 and X_train[c].nunique() > 1]
X_train = X_train[keep]
print("features:", len(feature_cols), "->", len(keep))

In [ ]:
cv = TimeSeriesSplit(n_splits=5)

def make(model, k=None):
    steps = [("imp", SimpleImputer(strategy="median")),
             ("var", VarianceThreshold(0.0)),    
             ("sc", StandardScaler())]
    if k:
        steps.append(("sel", SelectKBest(f_classif, k=k)))
    steps.append(("clf", model))
    return Pipeline(steps)

def evaluate(name, pipe, cv_obj=cv):
    res = cross_validate(pipe, X_train, y_train, cv=cv_obj,
                         scoring=["roc_auc", "average_precision"])
    print(f"{name:22s} ROC-AUC={res['test_roc_auc'].mean():.3f} (±{res['test_roc_auc'].std():.3f})  "
          f"PR-AUC={res['test_average_precision'].mean():.3f} (±{res['test_average_precision'].std():.3f})")

candidates = {
    "logreg, all features": make(LogisticRegression(class_weight="balanced", C=0.01, max_iter=2000)),
    "logreg, top 20":       make(LogisticRegression(class_weight="balanced", C=0.1, max_iter=2000), k=20),
    "logreg, top 50":       make(LogisticRegression(class_weight="balanced", C=0.1, max_iter=2000), k=50),
    "random forest":        make(RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                                        class_weight="balanced_subsample",
                                                        random_state=42, n_jobs=-1)),
}



In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score, recall_score,
                             precision_score, f1_score, confusion_matrix)

X_test, y_test = test[feature_cols][keep], test["label"]

final = candidates["logreg, top 50"]
final.fit(X_train, y_train)

proba = final.predict_proba(X_test)[:, 1]
pred = final.predict(X_test)

print("test fail rate:", round(y_test.mean(), 3), "(PR-AUC of random guessing is about this)")
print("ROC-AUC  :", round(roc_auc_score(y_test, proba), 3))
print("PR-AUC   :", round(average_precision_score(y_test, proba), 3))
print("recall   :", round(recall_score(y_test, pred), 3))
print("precision:", round(precision_score(y_test, pred, zero_division=0), 3))
print("F1       :", round(f1_score(y_test, pred), 3))
print(confusion_matrix(y_test, pred))